# Web Chatbot penelitian

Aplikasi ini dibangun dengan menggunakan arsitektur sederhana di mana **Google Scholar** (melalui library `scholarly`) bertindak sebagai sumber data pencarian. **Gemini API** bertindak sebagai otak LLM yang mensintesis jawaban dengan `temperature=0` (agar outputnya sangat faktual dan deterministik). **Streamlit** sebagai antarmuka penggunanya. **Ngrok** akan digunakan untuk mengekspos server lokal/Colab ke internet.


### 1. Persiapan Lingkungan (Instalasi Library)

In [ ]:
!pip install streamlit google-genai scholarly pyngrok -q

### 2. Script Aplikasi (Simpan sebagai `app.py`)

Script ini berisi logika antarmuka Streamlit, pencarian Google Scholar, dan pemanggilan Gemini API.

In [ ]:
%%writefile app.py
import streamlit as st
from google import genai
from google.genai import types
from scholarly import scholarly

# --- KONFIGURASI HALAMAN STREAMLIT ---\n
st.set_page_config(page_title="Chatbot Penelitian LLM v2", page_icon="📚", layout="wide")
st.title("📚 Chatbot Asisten Penelitian (SDK Baru)")
st.caption("Berbasis Gemini 1.5 Flash (google-genai) & Google Scholar")

# --- SIDEBAR: KONFIGURASI API KEY ---
with st.sidebar:
    st.header("Konfigurasi")
    api_key = st.text_input("Masukkan Google Gemini API Key:", type="password")
    if not api_key:
        st.warning("Silakan masukkan API Key untuk memulai.")

# --- FUNGSI PENCARIAN GOOGLE SCHOLAR ---
@st.cache_data(show_spinner=False)
def search_google_scholar(query, max_results=3):
    """Mencari publikasi di Google Scholar berdasarkan kata kunci."""
    results = []
    try:
        search_query = scholarly.search_pubs(query)
        for i in range(max_results):
            pub = next(search_query)
            title = pub.get('bib', {}).get('title', 'Tidak ada judul')
            abstract = pub.get('bib', {}).get('abstract', 'Tidak ada abstrak')
            author = pub.get('bib', {}).get('author', 'Tidak diketahui')
            year = pub.get('bib', {}).get('pub_year', 'Tahun tidak diketahui')

            results.append(f"Judul: {title}\nPenulis: {author}\nTahun: {year}\nAbstrak: {abstract}")
    except StopIteration:
        pass
    except Exception as e:
        return f"Error saat mengambil data dari Scholar: {e}"

    return "\n\n---\n\n".join(results) if results else "Tidak ada literatur yang ditemukan untuk kueri ini."

# --- FUNGSI LLM GEMINI (MENGGUNAKAN SDK GOOGLE-GENAI BARU) ---
def generate_response(prompt, scholar_context, api_key):
    """Menghasilkan respons dari Gemini berdasarkan konteks Scholar."""
    # 1. Inisialisasi Client Baru
    client = genai.Client(api_key=api_key)

    system_prompt = f"""
    Anda adalah asisten peneliti akademik yang sangat cerdas.
    Tugas Anda adalah menjawab pertanyaan pengguna berdasarkan literatur jurnal yang disediakan di bawah ini.
    Jika literatur tidak menjawab pertanyaan, katakan sejujurnya bahwa data tidak mencukupi, jangan mengarang informasi (halusinasi).

    [LITERATUR GOOGLE SCHOLAR]:
    {scholar_context}

    [PERTANYAAN PENGGUNA]:
    {prompt}
    """

    # 2. Panggilan API Menggunakan Format Config Baru
    response = client.models.generate_content(
        model='gemini-1.5-flash',
        contents=system_prompt,
        config=types.GenerateContentConfig(
            temperature=0.0, # Sangat faktual dan kaku
        )
    )
    return response.text

# --- LOGIKA CHAT INTERFACE ---
if "messages" not in st.session_state:
    st.session_state.messages = []

# Tampilkan riwayat obrolan
for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])

# Input dari pengguna
prompt = st.chat_input("Tanyakan sesuatu tentang topik penelitian Anda... (misal: Bagaimana pengaruh microplastic pada biota laut?)")
if prompt:
    # Tambahkan pertanyaan ke antarmuka
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    if not api_key:
        st.error("API Key belum dimasukkan. Silakan cek sidebar.")
    else:
        with st.chat_message("assistant"):
            with st.spinner("Mengekstrak kata kunci & mencari di Google Scholar..."):
                # Optimasi: Membersihkan kata tanya standar agar pencarian Scholar tidak zonk
                clean_query = prompt.lower()
                words_to_remove = ["apakah", "bagaimana", "mengapa", "apa", "dampak", "pengaruh", "dari", "pada", "ke", "di", "?", "jelaskan"]
                for word in words_to_remove:
                    clean_query = clean_query.replace(word, "")
                clean_query = " ".join(clean_query.split()) # Hapus spasi berlebih

                # Jika hasil pembersihan kosong, kembali ke prompt awal
                final_search_query = clean_query if clean_query else prompt

                # Jalankan pencarian Scholar
                scholar_context = search_google_scholar(final_search_query)
                st.info(f"🔍 **Kata kunci Scholar:** '{final_search_query}'\n\n**Hasil awal ditemukan:**\n" + scholar_context[:250] + "...")

            with st.spinner("Menyusun sintesis dengan Gemini..."):
                try:
                    # Panggil fungsi dengan parameter api_key yang baru
                    answer = generate_response(prompt, scholar_context, api_key)
                    st.markdown(answer)

                    # Simpan ke riwayat
                    st.session_state.messages.append({"role": "assistant", "content": answer})
                except Exception as e:
                    st.error(f"Terjadi kesalahan pada Gemini API: {e}")

### 3. Menjalankan Aplikasi dengan Ngrok

Penggunaan `pyngrok` bersama Streamlit digunakan jika menjalankan kode di lingkungan *cloud* (seperti Google Colab) atau ingin membagikan akses localhost Anda ke orang lain.

Buat file baru bernama `run_ngrok.py` (atau jalankan ini di sel berikutnya jika menggunakan Colab):

In [ ]:
from pyngrok import ngrok
import subprocess
import time
from google.colab import userdata

# Matikan tunnel lama jika ada
ngrok.kill()

# Ambil token dari Colab Secrets (Atau ganti langsung dengan string token Anda)
try:
    NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTH_TOKEN')
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
except Exception:
    print("⚠️ Gagal mengambil token dari Secrets Colab. Pastikan tab kunci 🔑 sudah diisi.")

print("Memulai Streamlit...")
process = subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501"])

# Tunggu server siap
time.sleep(3)

# Buka akses publik
public_url = ngrok.connect(8501)
print(f"✅ Aplikasi berhasil dijalankan!")
print(f"🌍 KLIK LINK INI UNTUK MEMBUKA CHATBOT: {public_url}")

try:
    process.wait()
except KeyboardInterrupt:
    print("Menutup server...")
    ngrok.kill()

### Cara Kerja Arsitektur Ini:

1. **RAG (Retrieval-Augmented Generation) Sederhana:** Ketika pengguna bertanya, sistem tidak langsung bertanya kepada LLM. Sistem akan mencari *keyword* dari pertanyaan tersebut ke Google Scholar terlebih dahulu.
2. **Konteks:** Ekstraksi judul, penulis, dan abstrak dari jurnal yang relevan lalu digabungkan menjadi satu teks panjang.
3. **`temperature=0`:** Prompt pengguna beserta konteks jurnal dikirimkan ke Gemini API. Karena `temperature` diset ke `0.0`, Gemini akan merespons secara sangat kaku, faktual, dan tidak kreatif, yang mana ini adalah praktik terbaik (Best Practice) untuk meminimalisir halusinasi dalam penelitian.
4. **Caching:** Fungsi `search_google_scholar` dibungkus dengan `@st.cache_data` agar pertanyaan yang sama tidak melakukan *scraping* berulang ke Google Scholar, yang bisa menyebabkan IP Anda diblokir sementara oleh Google.

Untuk menghentikan app dan menjalankan app berikutnya, jalankan cell di bawah:

In [ ]:
# Hentikan app sebelumnya sebelum menjalankan app baru
import os, signal
try:
    proc.terminate()
    print("App dihentikan.")
except:
    pass